## imports

In [1]:
import math
import random
import numpy as np
from itertools import combinations, product
import matplotlib.pyplot as plt
from time import perf_counter
from tqdm import tqdm

## computation

In [2]:
def canonical_necklace(coloring):
    """Lexicographically smallest rotation among the necklace and its
    reflection/color inversion. Reversing the necklace order maps every 
    extended-cut configuration to an equally-valid one with the same arc 
    multiset (red counts and lengths are preserved under reversal), so 
    colorings related by rotation/reflection/color inversion give equivalent 
    extended-cut problems."""
    m = len(coloring)
    best = None
    for seq1 in (tuple(coloring), tuple(c ^ 1 for c in coloring)):
        for seq2 in (seq1, tuple(reversed(seq1))):
            for i in range(m):
                rot = seq2[i:] + seq2[:i]
                if best is None or rot < best:
                    best = rot
    return best

In [3]:
def make_prefix(coloring):
    """Doubled prefix-sum array (as a plain list) so arcs that wrap around
    the circle can be queried with a single subtraction. The cumulative sum
    itself is computed with numpy; the result is converted to a list because
    repeated numpy-scalar indexing in the hot loop is slower than list
    indexing for arrays this small."""
    m = len(coloring)
    doubled = coloring + coloring
    prefix = np.concatenate([[0], np.cumsum(doubled)]).tolist()
    return prefix, m


def arcs_from_cuts(prefix, m, cutpoints):
    """Given doubled prefix sums and a set of cut points, return list of
    (red_count, length) for each resulting arc, going around the circle."""
    cuts = sorted(set(p % m for p in cutpoints))
    n = len(cuts)
    arcs = []
    for i in range(n):
        start = cuts[i]
        end = cuts[(i + 1) % n]
        if end <= start:
            end += m
        arcs.append((prefix[end] - prefix[start], end - start))
    return arcs

In [4]:
def can_partition(arcs, red_target, total_target):
    """Exact feasibility: can `arcs` be split into groups each summing to
    exactly (red_target, total_target)? Bitmask DP, O(3^n)."""
    n = len(arcs)
    sums = [(0, 0)] * (1 << n)
    for mask in range(1, 1 << n):
        lsb = mask & (mask - 1)
        idx = (mask ^ lsb).bit_length() - 1
        r0, t0 = sums[lsb]
        ar, at = arcs[idx]
        sums[mask] = (r0 + ar, t0 + at)

    dp = [False] * (1 << n)
    dp[0] = True
    for mask in range(1, 1 << n):
        sub = mask
        while sub:
            if sums[sub] == (red_target, total_target) and dp[mask ^ sub]:
                dp[mask] = True
                break
            sub = (sub - 1) & mask
    return dp[(1 << n) - 1]


def greedy_deviation(arcs, k, red_target, total_target):
    """Cheap (O(n log n + n*k)) heuristic: greedily assign arcs (largest red
    count first) to whichever of k bins is currently furthest below its red
    target, then sum |deviation| over bins. Zero deviation here implies an
    exact partition exists (each bin individually hits target), so this is a
    SOUND but INCOMPLETE feasibility check -- good enough to guide local
    search, but a nonzero result doesn't prove infeasibility."""
    bins_red = [0] * k
    bins_total = [0] * k
    for red, length in sorted(arcs, key=lambda a: -a[0]):
        j = min(range(k), key=lambda b: bins_red[b])
        bins_red[j] += red
        bins_total[j] += length
    return sum(abs(br - red_target) + abs(bt - total_target)
               for br, bt in zip(bins_red, bins_total))

In [5]:
def extended_cut_offsets(m):
    if m % 2 == 0:
        return (m // 2,)
    else:
        return (m // 2, m // 2 + 1)


def config_to_arcs(prefix, m, xs, offcombo):
    cutpoints = set()
    for x, o in zip(xs, offcombo):
        cutpoints.add(x % m)
        cutpoints.add((x + o) % m)
    if len(cutpoints) < 2:
        return None
    return arcs_from_cuts(prefix, m, cutpoints)

In [6]:
def _exhaustive_search_for_t(prefix, m, t, offsets, red_target, total_target, verbose=False):
    floor_half, ceil_half = m // 2, (m + 1) // 2
    
    # offsets must be the floor and/or ceil of m/2
    if m % 2:
        assert set(offsets) == {floor_half, ceil_half}, \
            f"Odd m={m} requires offsets=[{floor_half}, {ceil_half}], got {offsets}"
    else:
        assert set(offsets) == {floor_half}, \
            f"Even m={m} requires offsets=[{floor_half}], got {offsets}"

    half = ceil_half  # search primaries in [0, ceil(m/2))
    n_total = math.comb(half, t)
    for xs in tqdm(combinations(range(half), t), total=n_total, desc=f"t={t}", disable=not verbose):
        for offcombo in product(offsets, repeat=t):
            arcs = config_to_arcs(prefix, m, xs, offcombo)
            if arcs is not None and can_partition(arcs, red_target, total_target):
                return xs, offcombo, arcs
    return None


def _exhaustive_search_for_t_regular_start(prefix, m, t, offsets, red_target, total_target, verbose=False):
    k = m // total_target
    floor_half, ceil_half = m // 2, (m + 1) // 2

    # Same offset assertions as original
    if m % 2:
        assert set(offsets) == {floor_half, ceil_half}, \
            f"Odd m={m} requires offsets=[{floor_half}, {ceil_half}], got {offsets}"
    else:
        assert set(offsets) == {floor_half}, \
            f"Even m={m} requires offsets=[{floor_half}], got {offsets}"

    half = ceil_half

    # Fall back to plain exhaustive search if there aren't enough cut-pairs
    # to fill the evenly-spaced prefix
    if t < k:
        n_total = math.comb(half, t)
        for xs in tqdm(combinations(range(half), t), total=n_total, desc=f"t={t}", disable=not verbose):
            for offcombo in product(offsets, repeat=t):
                arcs = config_to_arcs(prefix, m, xs, offcombo)
                if arcs is not None and can_partition(arcs, red_target, total_target):
                    return xs, offcombo, arcs
        return None

    # t >= k: fix k evenly-spaced pairs, search the remaining t-k freely
    assert m % (2 * k) == 0, \
        f"m={m} must be divisible by 2*k={2 * k} for {k} evenly spaced pairs"
    spacing = m // (2 * k)
    # m divisible by 2k => m is even => floor_half == m//2
    # For any anchor in [0, spacing), all k primaries anchor+i*spacing
    # lie in [0, m//2), and each opposite is exactly primary + m//2. ✓
    fixed_offcombo = (floor_half,) * k
    anchors = range(spacing)
    n_free = t - k
    # len(free_pool) == half - k regardless of anchor, so total is constant
    n_total = math.comb(half - k, n_free)
    for anchor in anchors:
        fixed_xs = tuple(anchor + i * spacing for i in range(k))
        free_pool = [p for p in range(half) if p not in set(fixed_xs)]
        for free_xs in tqdm(
            combinations(free_pool, n_free),
            total=n_total,
            desc=f"t={t},k={k},anchor={anchor}",
            disable=not verbose,
        ):
            for free_offcombo in product(offsets, repeat=n_free):
                # Merge fixed and free pairs, sorted by position
                all_pairs = sorted(
                    list(zip(fixed_xs, fixed_offcombo)) + list(zip(free_xs, free_offcombo))
                )
                xs = tuple(x for x, _ in all_pairs)
                offcombo = tuple(o for _, o in all_pairs)
                arcs = config_to_arcs(prefix, m, xs, offcombo)
                if arcs is not None and can_partition(arcs, red_target, total_target):
                    return xs, offcombo, arcs
    return None


def _random_search_for_t(prefix, m, t, offsets, red_target, total_target,
                          samples, rng):
    if t > m:
        return None
    for _ in range(samples):
        xs = tuple(sorted(rng.sample(range(m), t)))
        offcombo = tuple(rng.choice(offsets) for _ in range(t))
        arcs = config_to_arcs(prefix, m, xs, offcombo)
        if arcs is not None and can_partition(arcs, red_target, total_target):
            return xs, offcombo, arcs
    return None


def _local_search_for_t(prefix, m, t, k, offsets, red_target, total_target,
                         restarts, steps, rng, exact_check_max_n=12):
    """Simulated annealing guided by greedy_deviation (fast but incomplete).
    If greedy_deviation never reaches 0, fall back to a single exact
    can_partition check on the best (lowest-deviation) configuration found,
    but only if it has few enough arcs to be affordable (O(3^n_arcs))."""
    if t > m:
        return None

    best = None  # (dev, xs, offcombo, arcs) -- lowest-deviation config seen

    for _ in range(restarts):
        xs = list(rng.sample(range(m), t))
        offcombo = [rng.choice(offsets) for _ in range(t)]
        arcs = config_to_arcs(prefix, m, xs, offcombo)
        cur_dev = greedy_deviation(arcs, k, red_target, total_target)
        if best is None or cur_dev < best[0]:
            best = (cur_dev, list(xs), list(offcombo), arcs)

        for step in range(steps):
            if cur_dev == 0:
                return tuple(xs), tuple(offcombo), arcs

            temperature = max(0.05, 1.0 - step / steps)
            i = rng.randrange(t)
            new_xs, new_offc = list(xs), list(offcombo)
            if rng.random() < 0.5:
                new_xs[i] = (new_xs[i] + rng.choice([-1, 1])) % m
                if len(set(new_xs)) != t:
                    continue
            else:
                new_offc[i] = rng.choice(offsets)

            new_arcs = config_to_arcs(prefix, m, new_xs, new_offc)
            if new_arcs is None:
                continue
            new_dev = greedy_deviation(new_arcs, k, red_target, total_target)

            if new_dev <= cur_dev or rng.random() < math.exp((cur_dev - new_dev) / temperature):
                xs, offcombo, cur_dev, arcs = new_xs, new_offc, new_dev, new_arcs
                if cur_dev < best[0]:
                    best = (cur_dev, list(xs), list(offcombo), arcs)

        if cur_dev == 0:
            return tuple(xs), tuple(offcombo), arcs

    # greedy_deviation never reached 0 anywhere. As a fallback, do a single
    # exact check on the best (lowest-deviation) configuration found -- this
    # catches partitions that exist but aren't reachable via the greedy
    # bin-assignment -- but only if it's cheap enough (O(3^n_arcs)).
    dev, bxs, boffc, barcs = best
    if dev > 0 and len(barcs) <= exact_check_max_n:
        if can_partition(barcs, red_target, total_target):
            return tuple(bxs), tuple(boffc), barcs

    return None

In [7]:
def min_extended_cuts(coloring, k, min_t=None, max_t=None, mode='exhaustive',
                       samples_per_t=2000, sa_restarts=20, sa_steps=100,
                       exact_check_max_n=12, seed=None, verbose=False):
    """Minimum number of extended cuts giving an exact k-way split.

    mode='exhaustive': guaranteed correct (within max_t), but expensive --- 
        roughly C(m,t) * |offsets|^t * 3^(2t) work at t.
    mode='exhaustive_regular_start': forces first k extended cuts to be 
        regularly spaced; a None result does NOT prove infeasibility.
    mode='random': samples random configurations per t; a None result does NOT 
        prove infeasibility.
    mode='local_search': simulated annealing on greedy_deviation, with a 
        single exact can_partition fallback check (when n_arcs <= 
        exact_check_max_n) on the best configuration found if greedy never 
        reaches 0. Successes are exact; a None result still does not prove 
        infeasibility for larger n_arcs.
    """
    m = len(coloring)
    r = sum(coloring)
    b = m - r
    if r % k != 0 or b % k != 0:
        raise ValueError(f"r={r}, b={b} not both divisible by k={k}; "
                          f"exact division is impossible regardless of cuts.")
    red_target = r // k
    total_target = m // k
    if min_t is None:
        min_t = 1
    if max_t is None:
        max_t = 2 * (k - 1)
    
    offsets = extended_cut_offsets(m)
    prefix, _ = make_prefix(coloring)
    rng = random.Random(seed)

    for t in range(min_t, max_t + 1):
        if mode == 'exhaustive':
            result = _exhaustive_search_for_t(prefix, m, t, offsets, red_target, total_target, verbose)
        elif mode == 'exhaustive_regular_start':
            result = _exhaustive_search_for_t_regular_start(
                prefix, m, t, offsets, red_target, total_target, verbose
            )
        elif mode == 'random':
            result = _random_search_for_t(prefix, m, t, offsets, red_target, total_target,
                                           samples_per_t, rng)
        elif mode == 'local_search':
            result = _local_search_for_t(prefix, m, t, k, offsets, red_target, total_target,
                                          sa_restarts, sa_steps, rng,
                                          exact_check_max_n=exact_check_max_n)
        else:
            raise ValueError(f"unknown mode: {mode}")

        if result is not None:
            xs, offcombo, arcs = result
            if verbose:
                print(f"  t={t}: (x,offset) pairs = {list(zip(xs, offcombo))}")
                print(f"        arcs (red,len) = {arcs}")
            return t, {"x_offset_pairs": list(zip(xs, offcombo)), "arcs": arcs}

    return None, None

In [8]:
def sweep_colorings(m, k, r, min_t, max_t, mode='exhaustive', dedupe=True,
                     coloring_sample=None, seed=None, **search_kwargs):
    """Check colorings of an m-bead necklace with exactly r red beads.

    coloring_sample=None : exhaustive over all C(m,r) colorings (with dedup).
    coloring_sample=N    : draw N random colorings instead (with dedup against
                            ones already seen). Use this when C(m,r) is too
                            large to enumerate.

    Returns (histogram of {min_t_or_None: count}, list of failed necklaces, 
    n_checked, n_total).
    """
    histogram = {}
    seen = set()

    if coloring_sample is None:
        iterator = combinations(range(m), r)
        n_total = math.comb(m, r)
    else:
        rng = random.Random(seed)
        iterator = (tuple(sorted(rng.sample(range(m), r))) for _ in range(coloring_sample))
        n_total = coloring_sample

    n_checked = 0
    failed_necklaces = []
    for positions in (pbar := tqdm(iterator, total=n_total)):
        coloring = [0] * m
        for p in positions:
            coloring[p] = 1
        if dedupe:
            canon = canonical_necklace(coloring)
            if canon in seen:
                continue
            seen.add(canon)
        t, _ = min_extended_cuts(coloring, k, min_t=min_t, max_t=max_t, mode=mode, **search_kwargs)
        if t is None:
            failed_necklaces.append(coloring)
        histogram[t] = histogram.get(t, 0) + 1
        n_checked += 1
        pbar.set_postfix(num_failed=len(failed_necklaces))
    return histogram, failed_necklaces, n_checked, n_total

In [9]:
def run_full_experiment(k_values, m_multiples, min_t=None, max_t=None,
                         mode='exhaustive', dedupe=True, coloring_sample=None,
                         seed=None, verbose=True, **search_kwargs):
    """Iterate over k in k_values; for each k, over m = j*k for j in
    m_multiples (j=1 has no valid r and is skipped); for each m, over valid
    r = i*k for i=1..j-1 (using color-swap symmetry r <-> m-r to halve the
    work: histogram(m,k,r) == histogram(m,k,m-r), since flipping every bead's
    color is a bijection between r-red and (m-r)-red colorings that preserves
    the minimum t); for each (m,k,r), over necklaces via sweep_colorings.

    max_t: either a fixed int, a callable k -> int, or None (defaults to
    2*(k-1)). Pass `lambda k: 2*(k-1) - 1` to test "is t < 2(k-1) always
    achievable?".

    Returns nested dict results[k][m][r] = (histogram, n_checked, n_total, dt)
    and prints a per-(k,m,r) summary line plus a final pass/fail table per k
    (pass = no coloring exceeded max_t, i.e. histogram has no None entry).
    """
    if not all(isinstance(k, int) and k > 1 for k in k_values):
        raise ValueError("k must be an integer greater than 1.")
    if not all(isinstance(mm, int) and mm > 1 for mm in m_multiples):
        raise ValueError("m multiple must be an integer greater than 1.")
    
    if min_t is None:
        min_t_fn = lambda k: 1
    elif callable(min_t):
        min_t_fn = min_t
    else:
        min_t_fn = lambda k: min_t
    
    if max_t is None:
        max_t_fn = lambda k: 2 * (k - 1) - 1
    elif callable(max_t):
        max_t_fn = max_t
    else:
        max_t_fn = lambda k: max_t

    results = {}
    summary = {}
    for k in k_values:
        results[k] = {}
        summary[k] = {"Mt": max_t_fn(k), "any_exceeded": False, "worst_t": None}
        for j in m_multiples:
            m = j * k
            results[k][m] = {}
            mt, Mt = min_t_fn(k), max_t_fn(k)
            for i in range(1, j):
                r = i * k
                if i > j - i and (j - i) * k in results[k][m]:
                    # color-swap symmetry: reuse histogram(m,k,(j-i)*k)
                    results[k][m][r] = results[k][m][(j - i) * k]
                    continue
                start = perf_counter()
                hist, failed_necklaces, checked, total = sweep_colorings(
                    m, k, r, mt, Mt, mode=mode, dedupe=dedupe,
                    coloring_sample=coloring_sample, seed=seed, **search_kwargs)
                dt = perf_counter() - start
                results[k][m][r] = {
                    "hist": hist, 
                    "failed_necklaces": failed_necklaces, 
                    "checked": checked, 
                    "total": total, 
                    "dt": dt
                }

                finite_ts = [tv for tv in hist if tv is not None]
                worst = max(finite_ts) if finite_ts else None
                n_fail = hist.get(None, 0)
                if n_fail > 0:
                    summary[k]["any_exceeded"] = True
                if worst is not None and (summary[k]["worst_t"] is None or worst > summary[k]["worst_t"]):
                    summary[k]["worst_t"] = worst

                if verbose:
                    worst_str = "failed" if n_fail > 0 else str(worst)
                    print(f"  k={k:2d} m={m:3d} r={r:3d} (j={j},i={i}): "
                          f"checked {checked}/{total}  worst-t={worst_str}  "
                          f"exceeded(>{Mt})={n_fail}  ({dt:.2f}s)")
                    
                    if failed_necklaces:
                        print("\nfailed necklaces:", *failed_necklaces, sep='\n\t')

    print("\n" + "=" * 70)
    print(f"{'k':>3}  {'k-1':>5}  {'2(k-1)':>7}  {'max_t tested':>12}  "
          f"{'worst t found':>13}  {'any exceeded?':>14}")
    for k in k_values:
        s = summary[k]
        worst_str = "failed" if s["any_exceeded"] else str(s["worst_t"])
        print(f"{k:>3}  {k-1:>5}  {2*(k-1):>7}  {s['Mt']:>12}  "
              f"{worst_str:>13}  {str(s['any_exceeded']):>14}")
    print("=" * 70)

    return results, summary

## visualization

In [10]:
def plot_necklace(coloring, x_offset_pairs=None, ax=None, figsize=(5, 5),
                   bead_colors=('#4472C4', '#C0504D'), show_indices=True,
                   ax_width_inches=None):
    """Visualize a circular necklace of 0/1 beads with matplotlib.
    coloring : list of 0/1.
    x_offset_pairs : optional list of (x, offset) extended-cut pairs, as
        returned in min_extended_cuts' details["x_offset_pairs"]. Each
        pair's two cut points (x and x+offset, mod m) are drawn as radial
        marks in a shared color; different pairs get different colors, so
        you can see which marks come from the same extended cut.
    ax, figsize : pass an existing Axes to draw on, or a figsize for a new
        figure (ignored if ax is given).
    bead_colors : fill color for beads with
        value 0 and value 1, respectively.
    show_indices : annotate each bead with its position index (0..m-1).
    ax_width_inches : width, in inches, that this Axes occupies on the
        figure. Used to convert data-unit sizes (bead radius, cut marks)
        to points so that fonts/lines scale correctly. If None, the full
        figure width is used (correct for a standalone single-axes
        figure); when placing multiple necklaces in a grid via
        plot_necklaces, this is set to each subplot's width instead.
    Layout is governed by a few constants defined at the top of the
    function body, all in terms of BEAD_RADIUS (the core size knob,
    in data units):
    STRING_TO_BEAD_RATIO : necklace string radius, as a multiple of
        BEAD_RADIUS.
    BEAD_PACKING_FRACTION : fraction of the circumferential gap between
        adjacent bead centers occupied by the bead diameter (caps bead
        size for small m so beads don't overlap).
    PLOT_MARGIN_RATIO : extra space around the necklace and cut marks, as
        a multiple of BEAD_RADIUS.
    FONT_SIZE_RATIO, CUT_LINEWIDTH_RATIO, CUT_LENGTH_RATIO : digit font 
        size, cut-mark line width, and cut-mark length, each as a fraction 
        of the bead radius once converted to points.
    CUT_COLOR_SATURATION, CUT_COLOR_VALUE : HSV saturation/value used for
        cut-mark colors, chosen to match the muted tone of bead_colors.
    POINTS_PER_INCH : standard typographic conversion (72 points/inch),
        used to convert data-unit sizes to font/line sizes in points.
    Returns the matplotlib Axes (does not call plt.show()).
    """
    BEAD_RADIUS = 0.06
    STRING_WIDTH = 1
    STRING_TO_BEAD_RATIO = 10
    BEAD_PACKING_FRACTION = 0.8
    PLOT_MARGIN_RATIO = 1.5
    FONT_SIZE_RATIO = 0.8
    CUT_LINEWIDTH_RATIO = 0.2
    CUT_LENGTH_RATIO = 2.8
    CUT_COLOR_SATURATION = 0.6
    CUT_COLOR_VALUE = 0.75
    POINTS_PER_INCH = 72
    m = len(coloring)
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)
    string_radius = BEAD_RADIUS * STRING_TO_BEAD_RATIO
    bead_radius = min(BEAD_PACKING_FRACTION * np.pi * string_radius / m, BEAD_RADIUS)
    half_width = string_radius + bead_radius + PLOT_MARGIN_RATIO * BEAD_RADIUS
    if ax_width_inches is None:
        ax_width_inches = ax.figure.get_size_inches()[0]
    points_per_unit = ax_width_inches / (2 * half_width) * POINTS_PER_INCH
    bead_radius_points = bead_radius * points_per_unit
    fontsize = bead_radius_points * FONT_SIZE_RATIO
    cut_linewidth = bead_radius_points * CUT_LINEWIDTH_RATIO
    def bead_angle(i):
        return np.pi / 2 - 2 * np.pi * i / m
    ax.plot(0, 0, 'o', color='lightgray', markersize=2 * STRING_WIDTH)
    ax.add_patch(plt.Circle((0, 0), 
                            string_radius, 
                            fill=False, 
                            color='lightgray', 
                            linewidth=STRING_WIDTH, 
                            zorder=1)
                )
    for i, c in enumerate(coloring):
        a = bead_angle(i)
        x, y = string_radius * np.cos(a), string_radius * np.sin(a)
        ax.add_patch(plt.Circle((x, y), bead_radius, facecolor=bead_colors[c],
                                 edgecolor='black', zorder=2))
        if show_indices:
            ax.text(x, y, str(i), ha='center', va='center', fontsize=fontsize, zorder=3,
                    color='white')
    if x_offset_pairs:
        n_pairs = len(x_offset_pairs)
        hues = np.linspace(0, 1, n_pairs, endpoint=False)
        hsv = np.stack([hues, np.full(n_pairs, CUT_COLOR_SATURATION), np.full(n_pairs, CUT_COLOR_VALUE)], axis=1)
        cut_colors = plt.matplotlib.colors.hsv_to_rgb(hsv)
        for idx, (x, offset) in enumerate(x_offset_pairs):
            color = cut_colors[idx]
            for p in (x % m, (x + offset) % m):
                a = np.pi / 2 - 2 * np.pi * (p - 0.5) / m
                inner = string_radius - bead_radius * 0.5 * CUT_LENGTH_RATIO
                outer = string_radius + bead_radius * 0.5 * CUT_LENGTH_RATIO
                ax.plot([inner * np.cos(a), outer * np.cos(a)],
                        [inner * np.sin(a), outer * np.sin(a)],
                        color=color, linewidth=cut_linewidth, zorder=4)
    ax.set_xlim(-half_width, half_width)
    ax.set_ylim(-half_width, half_width)
    ax.set_aspect('equal')
    ax.axis('off')
    return ax


def plot_necklaces(colorings, x_offset_pairs_list=None, ncols=None,
                    subplot_size=3, bead_colors=('#4472C4', '#C0504D'),
                    show_indices=True):
    """Visualize a grid of circular necklaces with matplotlib.

    colorings : list of colorings, each a list of 0/1 (as accepted by
        plot_necklace's coloring argument). One subplot is drawn per
        coloring.
    x_offset_pairs_list : optional list, the same length as colorings,
        where each entry is either None or a list of (x, offset) pairs
        for that necklace (as accepted by plot_necklace's
        x_offset_pairs argument). If None, no cut marks are drawn on any
        necklace.
    ncols : number of columns in the grid. Defaults to ceil(sqrt(n)),
        giving a roughly square grid.
    subplot_size : side length, in inches, of each (square) subplot.
        Controls both the figure size and the conversion of data-unit
        sizes (bead radius, cut marks, index font) to points.
    bead_colors, show_indices : passed through to plot_necklace for
        every subplot.

    Any grid cells beyond len(colorings) are left empty (axes turned
    off).

    Returns (fig, axes), the matplotlib Figure and a 2D array of Axes
    of shape (nrows, ncols).
    """
    n = len(colorings)
    if x_offset_pairs_list is None:
        x_offset_pairs_list = [None] * n
    if len(x_offset_pairs_list) != n:
        raise ValueError("x_offset_pairs_list must have the same length as colorings")

    if ncols is None:
        ncols = int(np.ceil(np.sqrt(n)))
    ncols = max(ncols, 1)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(subplot_size * ncols, subplot_size * nrows))
    axes = np.atleast_1d(axes).reshape(nrows, ncols)

    for idx in range(nrows * ncols):
        r, c = divmod(idx, ncols)
        ax = axes[r, c]
        if idx < n:
            plot_necklace(colorings[idx], x_offset_pairs=x_offset_pairs_list[idx],
                           ax=ax, bead_colors=bead_colors, show_indices=show_indices,
                           ax_width_inches=subplot_size)
        else:
            ax.axis('off')

    fig.tight_layout()
    return fig, axes

## experiments

In [11]:
seed = random.randint(0, 10**6)

results, _ = run_full_experiment(
    k_values=[3],
    m_multiples=(3,4),
    min_t=lambda k: k,
    mode="exhaustive",
    seed=seed
)

100%|███████████████████████████████████████████████| 84/84 [00:00<00:00, 2098.73it/s, num_failed=0]


  k= 3 m=  9 r=  3 (j=3,i=1): checked 7/84  worst-t=3  exceeded(>3)=0  (0.09s)


100%|█████████████████████████████████████████████| 220/220 [00:00<00:00, 2800.78it/s, num_failed=0]


  k= 3 m= 12 r=  3 (j=4,i=1): checked 12/220  worst-t=3  exceeded(>3)=0  (0.09s)


100%|█████████████████████████████████████████████| 924/924 [00:00<00:00, 3434.66it/s, num_failed=0]


  k= 3 m= 12 r=  6 (j=4,i=2): checked 35/924  worst-t=3  exceeded(>3)=0  (0.28s)

  k    k-1   2(k-1)  max_t tested  worst t found   any exceeded?
  3      2        4             3              3           False


In [12]:
def save_necklaces(lsts, parameters_id):
    strs = ["".join("R" if x else "B" for x in lst) for lst in lsts]
    with open(f"data/necklaces_{parameters_id}.txt", "w") as f:
        f.write("\n".join(strs) + "\n")

In [13]:
# failed_necklaces_k3m18r9 = results[3][18][9]["failed_necklaces"]
# save_necklaces(failed_necklaces_k3m18r9, "k3m18r9")
# plot_necklaces(failed_necklaces_k3m18r9)
# plt.savefig("images/necklaces_k3m18r9.pdf", transparent=True)
# plt.show()

In [14]:
# failed_necklaces_k3m24r9 = results[3][24][9]["failed_necklaces"]
# save_necklaces(failed_necklaces_k3m24r9, "k3m24r9")
# plot_necklaces(failed_necklaces_k3m24r9)
# plt.savefig("images/necklaces_k3m24r9.pdf", transparent=True)
# plt.show()

In [15]:
# failed_necklaces_k3m24r12 = results[3][24][12]["failed_necklaces"]
# save_necklaces(failed_necklaces_k3m24r12, "k3m24r12")
# plot_necklaces(failed_necklaces_k3m24r12)
# plt.savefig("images/necklaces_k3m24r12.pdf", transparent=True)
# plt.show()

### testing specific necklaces

In [16]:
k = 5
coloring = [1]*5 + [0]*5
min_extended_cuts(coloring, k=k, max_t=2*(k-1)-1, verbose=True);

t=4:  80%|███████████████████████████████████████████████▏           | 4/5 [00:00<00:00, 233.64it/s]

  t=4: (x,offset) pairs = [(1, 5), (2, 5), (3, 5), (4, 5)]
        arcs (red,len) = [(1, 1), (1, 1), (1, 1), (1, 2), (0, 1), (0, 1), (0, 1), (1, 2)]


Specific necklaces with k=5, m=30, and r=15 that can be cut using at most 2(k-1)=7 cuts:
- [1]*13 + [0]*7 + [1,0,1] + [0]*7 (6 cuts)
- [1]*11 + [0]*7 + [1,1,0,1,1] + [0]*7 (6 cuts)
- [1]*11 + [0]*6 + [1,0,1,0,1,0,1] + [0]*6 (5 cuts)
- [1]*9 + [0]*7 + [1,1,1,0,1,1,1] + [0]*7 (5 cuts)
- [1]*11 + [0,1] + [0]*6 + [1,0,1] + [0]*6 + [1,0] (5 cuts)
- [1]*9 + [0,1] + [0]*6 + [1,1,0,1,1] + [0]*6 + [1,0] (6 cuts)
- [1]*5 + [0]*3 + [1,1,0,1,1] + [0,0,0,1,0,1,0,0,0] + [1,1,0,1,1] + [0]*3 (4 cuts)